# 05 — Model Diagnostics

This notebook performs residual diagnostics for the fitted GARCH model:
- Ljung-Box test on standardized residuals
- ARCH-LM test on squared standardized residuals
- ACF/PACF plots
- Distribution analysis of standardized residuals

**Pipeline step 6 of 7.**

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from src.data_loader import load_processed_data
from src.preprocessing import prepare_returns_for_garch
from src.model_builder import GARCHModelFactory, ModelSpec
from src.diagnostics import (
    ljung_box_test, arch_lm_test,
    plot_acf_pacf, plot_garch_diagnostics, durbin_watson_test
)

%matplotlib inline

## 5.1 Fit Baseline GARCH(1,1)

In [ ]:
df = load_processed_data()
returns_pct = prepare_returns_for_garch(df["log_return"].dropna())

factory = GARCHModelFactory(returns_pct)
spec = ModelSpec("GARCH", p=1, q=1, distribution="normal")
fit_result = factory.fit(spec)

## 5.2 Extract Standardized Residuals

In [ ]:
std_resid = fit_result.result.std_resid.dropna()
print(f"Standardized residuals: {len(std_resid)} observations")
print(std_resid.describe())

## 5.3 Ljung-Box Test on Standardized Residuals

H0: No autocorrelation in standardized residuals.

In [ ]:
lb_stat, lb_pval, lb_passed = ljung_box_test(std_resid, lags=10, verbose=True)

## 5.4 ARCH-LM Test on Squared Standardized Residuals

H0: No remaining ARCH effects in standardized residuals.

In [ ]:
lm_stat, lm_pval, lm_passed = arch_lm_test(std_resid, lags=10, verbose=True)

## 5.5 ACF/PACF of Standardized Residuals

In [ ]:
fig = plot_acf_pacf(std_resid, lags=20, title_prefix="Standardized Residuals")
plt.savefig("../results/figures/acf_pacf_std_resid.png", dpi=150)
plt.show()

## 5.6 Full Diagnostic Panel

In [ ]:
fig = plot_garch_diagnostics(fit_result.result, figsize=(14, 10))
plt.savefig("../results/figures/garch_diagnostics.png", dpi=150)
plt.show()

## 5.7 Durbin-Watson Statistic

In [ ]:
dw = durbin_watson_test(std_resid)
print(f"Durbin-Watson: {dw:.4f} (ideal: ~2.0)")